In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import plotly.express as px

In [2]:


ROOT = Path(os.environ.get("FMRIDECOMP_OUTPUTS",
                           "/project/6008063/tamires/DecomposingfMRI/outputs"))
ATLAS = "harvardoxford"
WINDOW_S = 30                     # dfc has one more partition level than activation
SELECT = {"camcan": ["*"], "cneuromod": ["*"], "ds002837": ["*"]}
N_COMPONENTS = 5

# Everything in a dfc file that is not an edge.
QC = ["window_id", "start_tr", "start_s", "stimulus_start_s", "stimulus_end_s",
      "n_tr_nominal", "n_tr_available", "n_tr_effective", "frac_good_frames",
      "crosses_run_boundary", "crosses_clip_boundary", "rank_deficient",
      "ses", "run", "acq", "run_key"]
KEYS = ["atlas", "window_s", "cohort", "task", "sub"]

paths = [p
         for cohort, subs in SELECT.items()
         for sub in subs
         for p in sorted(ROOT.glob(
             f"dfc/atlas={ATLAS}/window_s={WINDOW_S}/cohort={cohort}/task=*/sub={sub}/data.parquet"))]
print(f"{len(paths)} shard(s)")

# The partition keys are directory names, so put them back as columns.
act = pd.concat(
    [pd.read_parquet(p).assign(**dict(s.split("=", 1) for s in p.parts if "=" in s))
     for p in paths],
    ignore_index=True)

parcels = [c for c in act.columns if "__" in c]      # edges: NodeA__NodeB
print(f"{len(act):,} windows x {len(parcels)} edges, "
      f"{act['sub'].nunique()} subject(s)")
act.head(3)

973 shard(s)
171,619 windows x 6105 edges, 738 subject(s)


,window_id,start_tr,start_s,stimulus_start_s,stimulus_end_s,n_tr_nominal,n_tr_available,n_tr_effective,frac_good_frames,crosses_run_boundary,...,Right_Pallidum__Right_Amygdala,Right_Pallidum__Right_Accumbens,Right_Hippocampus__Right_Amygdala,Right_Hippocampus__Right_Accumbens,Right_Amygdala__Right_Accumbens,atlas,window_s,cohort,task,sub
0,0,0,0.00,0.0,30.0,12,13,13,1.0,False,...,-0.444850,-0.753876,0.861812,0.521321,0.438034,harvardoxford,30,camcan,Movie,CC110033
1,1,3,7.41,6.0,36.0,12,12,12,1.0,False,...,-0.388300,-0.588657,0.792628,0.725968,0.746970,harvardoxford,30,camcan,Movie,CC110033
2,2,5,12.35,12.0,42.0,12,13,13,1.0,False,...,-0.045148,-0.397914,0.736807,-0.011130,0.485851,harvardoxford,30,camcan,Movie,CC110033
